<a href="https://colab.research.google.com/github/drsrishtiagrawal/drsrishtiagrawal/blob/main/GroupDNA_SRISHTI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# ==========================================
# Feature 1: The Chat Parser (Foolproof Version)
# GroupDNA - Unlox Academy Minor Project
# ==========================================

import os

def robust_parse_chat(file_path):
    parsed_messages = []
    system_count = 0
    media_count = 0
    deleted_count = 0

    # Check if file exists first
    if not os.path.exists(file_path):
        print(f"❌ Error: The file was not found at '{file_path}'.")
        print("💡 Tip: Make sure you dragged and dropped 'hostel_bois.txt' into the Colab file sidebar on the left.")
        return []

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                line_str = line.strip()
                if not line_str:
                    continue

                comma_idx = line_str.find(',')
                dash_idx = line_str.find(' - ')

                if comma_idx == -1 or dash_idx == -1:
                    system_count += 1
                    continue

                timestamp = line_str[:dash_idx].strip()
                content_after_dash = line_str[dash_idx + 3:].strip()

                colon_idx = content_after_dash.find(':')

                if colon_idx == -1:
                    system_count += 1
                    continue

                sender = content_after_dash[:colon_idx].strip()
                message_text = content_after_dash[colon_idx + 1:].strip()

                if message_text == '<Media omitted>':
                    media_count += 1
                elif message_text == 'This message was deleted':
                    deleted_count += 1

                message_dict = {
                    'timestamp': timestamp,
                    'sender': sender,
                    'text': message_text
                }
                parsed_messages.append(message_dict)

        print(f"✅ Successfully parsed {len(parsed_messages)} real messages!")
        print(f"📌 Skipped System Messages: {system_count}")
        print(f"📌 Media Omitted Stubs: {media_count}")
        print(f"📌 Deleted Messages: {deleted_count}")

    except Exception as e:
        print(f"⚠️ An unexpected error occurred while reading the file: {e}")

    return parsed_messages

# Run the parser
file_path = '/hostel_bois.txt'
chat_data = robust_parse_chat(file_path)

# Preview if successful
if chat_data:
    print("\n--- Parser Preview (First 3 Messages) ---")
    for m in chat_data[:3]:
        print(m)

✅ Successfully parsed 3174 real messages!
📌 Skipped System Messages: 4
📌 Media Omitted Stubs: 32
📌 Deleted Messages: 15

--- Parser Preview (First 3 Messages) ---
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'scene fix'}
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'haan'}
{'timestamp': '01/04/24, 01:18', 'sender': 'Rahul', 'text': 'kya scene'}


In [11]:
# ==========================================
# Feature 2: Group Overview & Per-Person Stats
# GroupDNA - Unlox Academy Minor Project
# ==========================================

from datetime import datetime

def generate_group_overview(chat_data):
    if not chat_data:
        print("⚠️ No chat data found. Please make sure Feature 1 ran successfully!")
        return

    total_messages = len(chat_data)

    # Extract unique participants
    participants = set(msg['sender'] for msg in chat_data)
    total_participants = len(participants)

    # Compute per-person message counts (using standard dict without collections.Counter)
    message_counts = {}
    for msg in chat_data:
        sender = msg['sender']
        message_counts[sender] = message_counts.get(sender, 0) + 1

    # Sort participants by message count from highest to lowest
    sorted_participants = sorted(message_counts.items(), key=lambda x: x[1], reverse=True)

    # Compute date range using datetime for accuracy
    # Timestamp format: 'DD/MM/YY, HH:MM'
    first_ts_str = chat_data[0]['timestamp'].split(',')[0].strip()
    last_ts_str = chat_data[-1]['timestamp'].split(',')[0].strip()

    date_format = '%d/%m/%y'
    first_date = datetime.strptime(first_ts_str, date_format)
    last_date = datetime.strptime(last_ts_str, date_format)

    # Total days inclusive
    total_days = (last_date - first_date).days + 1

    # Human-readable date formatting
    first_date_readable = first_date.strftime('%d %B %Y')
    last_date_readable = last_date.strftime('%d %B %Y')

    # Print the formatted report section
    print("=" * 60)
    print("GROUP OVERVIEW")
    print("=" * 60)
    print(f"Group               : Hostel Bois 4ever")
    print(f"Period              : {first_date_readable} to {last_date_readable} ({total_days} days)")
    print(f"Total messages      : {total_messages:,}")
    print(f"Participants        : {total_participants}")
    print("-" * 60)
    print("MESSAGES PER PERSON")
    print("-" * 60)

    for sender, count in sorted_participants:
        percentage = (count / total_messages) * 100
        print(f"{sender:<18} : {count:<5} ({percentage:.1f}%)")

    print("=" * 60)

# Run Feature 2
generate_group_overview(chat_data)

GROUP OVERVIEW
Group               : Hostel Bois 4ever
Period              : 01 April 2024 to 30 May 2024 (60 days)
Total messages      : 3,174
Participants        : 6
------------------------------------------------------------
MESSAGES PER PERSON
------------------------------------------------------------
Rahul              : 953   (30.0%)
Priya              : 718   (22.6%)
Neha               : 635   (20.0%)
Aman               : 490   (15.4%)
Karan              : 354   (11.2%)
Vikas              : 24    (0.8%)


In [12]:
# ==========================================
# Feature 3: Busiest Day & Hour
# GroupDNA - Unlox Academy Minor Project
# ==========================================

from datetime import datetime

def analyze_busiest_times(chat_data):
    if not chat_data:
        print("⚠️ No chat data found. Please make sure Feature 1 ran successfully!")
        return

    # Dictionaries to aggregate counts
    daily_counts = {}
    hourly_counts = {}

    for msg in chat_data:
        timestamp_str = msg['timestamp'] # Format: 'DD/MM/YY, HH:MM'

        # Split date and time parts
        date_part, time_part = timestamp_str.split(',')
        date_part = date_part.strip()
        time_part = time_part.strip()

        # Extract the hour (e.g., '23:14' -> '23')
        hour_part = time_part.split(':')[0]

        # 1. Aggregate by Day
        daily_counts[date_part] = daily_counts.get(date_part, 0) + 1

        # 2. Aggregate by Hour (00 to 23)
        hourly_counts[hour_part] = hourly_counts.get(hour_part, 0) + 1

    # Find the busiest day (maximum message count)
    busiest_date_str = max(daily_counts, key=daily_counts.get)
    busiest_day_count = daily_counts[busiest_date_str]

    # Format the busiest date nicely (e.g., '01/04/24' -> '01 April 2024')
    busiest_date_obj = datetime.strptime(busiest_date_str, '%d/%m/%y')
    busiest_date_formatted = busiest_date_obj.strftime('%d %B %Y')

    # Find the peak hour (maximum message count)
    peak_hour_str = max(hourly_counts, key=hourly_counts.get)
    peak_hour_count = hourly_counts[peak_hour_str]

    # Format hour nicely (e.g., '23' -> '11:00 PM - 12:00 AM')
    hour_int = int(peak_hour_str)
    if hour_int == 0:
        hour_formatted = "12:00 AM - 01:00 AM"
    elif hour_int < 12:
        hour_formatted = f"{hour_int:02d}:00 AM - {(hour_int+1):02d}:00 AM"
    elif hour_int == 12:
        hour_formatted = "12:00 PM - 01:00 PM"
    else:
        hour_formatted = f"{(hour_int-12):02d}:00 PM - {((hour_int-12)+1):02d}:00 PM"

    # Print the report section
    print("=" * 60)
    print("PEAK ACTIVITY ANALYSIS")
    print("=" * 60)
    print(f"🔥 Busiest Day      : {busiest_date_formatted} ({busiest_day_count:,} messages)")
    print(f"⏰ Peak Hour        : {hour_formatted} ({peak_hour_count:,} messages)")
    print("=" * 60)

# Run Feature 3
analyze_busiest_times(chat_data)

PEAK ACTIVITY ANALYSIS
🔥 Busiest Day      : 04 May 2024 (76 messages)
⏰ Peak Hour        : 06:00 PM - 07:00 PM (248 messages)


In [13]:
# ==========================================
# Feature 4: NumPy Activity Heatmap
# GroupDNA - Unlox Academy Minor Project
# ==========================================

import numpy as np

def generate_activity_heatmap(chat_data):
    if not chat_data:
        print("⚠️ No chat data found. Please make sure Feature 1 ran successfully!")
        return

    # 1. Identify all unique participants and sort them consistently
    participants = sorted(list(set(msg['sender'] for msg in chat_data)))

    # Map participant name to a row index (0 to N-1)
    participant_to_idx = {name: idx for idx, name in enumerate(participants)}

    num_participants = len(participants)

    # 2. Initialize a 6x24 NumPy matrix (Rows = participants, Columns = hours 0-23)
    heatmap_matrix = np.zeros((num_participants, 24), dtype=int)

    # 3. Populate the matrix
    for msg in chat_data:
        sender = msg['sender']
        timestamp_str = msg['timestamp'] # Format: 'DD/MM/YY, HH:MM'

        try:
            # Extract hour part
            time_part = timestamp_str.split(',')[1].strip()
            hour_str = time_part.split(':')[0]
            hour = int(hour_str)

            row_idx = participant_to_idx[sender]
            heatmap_matrix[row_idx, hour] += 1
        except Exception:
            continue # Skip malformed timestamps safely

    # 4. Define text-art density characters based on relative message frequency
    def get_density_char(val, max_val):
        if val == 0:
            return '.'
        elif val <= max_val * 0.25:
            return ','
        elif val <= max_val * 0.60:
            return '-'
        else:
            return '█'

    max_msg_count = heatmap_matrix.max() if heatmap_matrix.size > 0 else 1

    # 5. Print the Heatmap Report
    print("=" * 80)
    print("ACTIVITY HEATMAP (PARTICIPANTS vs. HOURS OF THE DAY)")
    print("=" * 80)
    print("Legend: [.] None  [,] Low  [-] Medium  [█] High\n")

    # Print Hour Header row (00 to 23)
    hour_header = "Participant       | " + " ".join([f"{h:02d}" for h in range(24)])
    print(hour_header)
    print("-" * len(hour_header))

    # Print each participant's row
    for idx, participant in enumerate(participants):
        row_str = f"{participant:<17} | "
        for hour in range(24):
            val = heatmap_matrix[idx, hour]
            char = get_density_char(val, max_msg_count)
            row_str += f" {char} "
        print(row_str)

    print("=" * 80)

# Run Feature 4
generate_activity_heatmap(chat_data)

ACTIVITY HEATMAP (PARTICIPANTS vs. HOURS OF THE DAY)
Legend: [.] None  [,] Low  [-] Medium  [█] High

Participant       | 00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23
-------------------------------------------------------------------------------------------
Aman              |  -  █  █  -  █  .  .  .  .  .  .  .  .  .  ,  ,  ,  ,  ,  ,  ,  ,  .  - 
Karan             |  .  .  .  .  .  .  .  ,  ,  ,  ,  ,  -  ,  -  -  -  -  ,  -  ,  ,  ,  , 
Neha              |  .  .  .  .  .  ,  ,  ,  -  -  -  ,  -  -  -  ,  -  -  -  -  -  -  -  - 
Priya             |  .  .  .  .  .  .  ,  ,  -  █  -  -  -  -  -  -  -  -  -  -  -  -  ,  , 
Rahul             |  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  ,  -  -  -  -  █  -  █  █  -  █  -  - 
Vikas             |  .  .  .  .  .  .  .  ,  ,  ,  ,  .  ,  ,  .  ,  ,  ,  ,  ,  ,  ,  ,  , 


In [14]:
# ==========================================
# Feature 5: Top Words Frequency & Bar Chart
# GroupDNA - Unlox Academy Minor Project
# ==========================================

def analyze_top_words(chat_data):
    if not chat_data:
        print("⚠️ No chat data found. Please make sure Feature 1 ran successfully!")
        return

    # Define a comprehensive stop words set to filter out noise
    stop_words = {
        'the', 'a', 'an', 'and', 'in', 'to', 'of', 'it', 'is', 'i', 'that',
        'you', 'he', 'was', 'for', 'on', 'are', 'as', 'with', 'his', 'they',
        'at', 'be', 'this', 'have', 'from', 'or', 'had', 'by', 'not', 'word',
        'but', 'what', 'some', 'we', 'can', 'out', 'other', 'were', 'all',
        'there', 'when', 'up', 'use', 'your', 'how', 'said', 'an', 'each',
        'she', 'which', 'do', 'their', 'time', 'if', 'will', 'way', 'about',
        'many', 'then', 'them', 'would', 'write', 'like', 'so', 'these', 'her',
        'long', 'make', 'thing', 'see', 'him', 'two', 'has', 'look', 'more',
        'day', 'could', 'go', 'come', 'did', 'my', 'no', 'most', 'pdf', 'media',
        'omitted', 'ye', 'ki', 'hain', 'ho', 'ko', 'se', 'ka', 'hain', 'bhi',
        'tha', 'kya', 'hai', 'toh', 'me', 'nahi', 'pe', 'yeh'
    }

    word_counts = {}

    # Punctuation marks to strip manually (since regex is forbidden)
    punctuation = '''!()-[]{};:'"\\,<>./?@#$%^&*_~+`='''

    for msg in chat_data:
        text = msg['text']

        # Skip media omitted or deleted messages
        if text == '<Media omitted>' or text == 'This message was deleted':
            continue

        # Convert to lowercase
        text_lower = text.lower()

        # Remove punctuation by replacing each with a space
        cleaned_text = ""
        for char in text_lower:
            if char in punctuation:
                cleaned_text += " "
            else:
                cleaned_text += char

        # Split into words by whitespace
        words = cleaned_text.split()

        # Filter stop words and short tokens, then aggregate
        for word in words:
            word = word.strip()
            if len(word) > 2 and word not in stop_words:
                word_counts[word] = word_counts.get(word, 0) + 1

    # Sort words by frequency in descending order
    sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
    top_10_words = sorted_words[:10]

    if not top_10_words:
        print("⚠️ No valid words found for frequency analysis.")
        return

    # Find maximum count for proportional bar scaling
    max_count = top_10_words[0][1]
    max_bar_length = 25  # Max visual length of the bar

    # Print the report section
    print("=" * 60)
    print("TOP 10 MOST FREQUENT WORDS")
    print("=" * 60)

    for word, count in top_10_words:
        # Calculate proportional bar length
        bar_len = int((count / max_count) * max_bar_length)
        bar = '█' * bar_len
        print(f"{word:<12} : {count:<5} {bar}")

    print("=" * 60)

# Run Feature 5
analyze_top_words(chat_data)

TOP 10 MOST FREQUENT WORDS
guys         : 318   █████████████████████████
today        : 292   ██████████████████████
just         : 208   ████████████████
everyone     : 203   ███████████████
telling      : 179   ██████████████
bhai         : 160   ████████████
anyone       : 157   ████████████
one          : 157   ████████████
started      : 150   ███████████
scene        : 145   ███████████


In [15]:
# ==========================================
# Feature 6: Response Speed & Silent Streaks
# GroupDNA - Unlox Academy Minor Project
# ==========================================

from datetime import datetime

def analyze_response_speeds_and_streaks(chat_data):
    if not chat_data:
        print("⚠️ No chat data found. Please make sure Feature 1 ran successfully!")
        return

    date_format = '%d/%m/%y, %H:%M'

    # Track response gaps (in minutes) per person
    person_gaps = {}

    # Track message timestamps per person to find silent streaks (days)
    person_message_dates = {}

    prev_msg = None

    for msg in chat_data:
        sender = msg['sender']
        timestamp_str = msg['timestamp']

        try:
            current_dt = datetime.strptime(timestamp_str, date_format)
        except Exception:
            continue

        # 1. Calculate response speed if the sender changed
        if prev_msg is not None:
            prev_sender = prev_msg['sender']
            prev_dt = prev_msg['parsed_dt']

            # If a different person replies, calculate the time gap
            if sender != prev_sender:
                diff_seconds = (current_dt - prev_dt).total_seconds()
                diff_minutes = diff_seconds / 60.0

                # Filter out unrealistic gaps (> 24 hours) for active conversation speed
                if 0 <= diff_minutes <= 1440:
                    if sender not in person_gaps:
                        person_gaps[sender] = []
                    person_gaps[sender].append(diff_minutes)

        # Attach parsed datetime for iteration
        msg['parsed_dt'] = current_dt
        prev_msg = msg

        # 2. Track dates for silent streaks
        date_only = current_dt.date()
        if sender not in person_message_dates:
            person_message_dates[sender] = set()
        person_message_dates[sender].add(date_only)

    # Compute average response times
    avg_response_times = {}
    for sender, gaps in person_gaps.items():
        if gaps:
            avg_response_times[sender] = sum(gaps) / len(gaps)

    # Compute longest silent streaks (consecutive days without messaging)
    # First, get the overall group start and end date span
    all_dates = sorted(list(set(msg['parsed_dt'].date() for msg in chat_data)))

    ghost_streaks = {}
    if all_dates:
        min_date = all_dates[0]
        max_date = all_dates[-1]

        # Generate full list of days in range
        total_span_days = []
        curr = min_date
        while curr <= max_date:
            total_span_days.append(curr)
            # increment day manually without timedelta if needed, or import timedelta
            from datetime import timedelta
            curr += timedelta(days=1)

        for sender, active_dates in person_message_dates.items():
            max_streak = 0
            current_streak = 0
            for day in total_span_days:
                if day not in active_dates:
                    current_streak += 1
                    if current_streak > max_streak:
                        max_streak = current_streak
                else:
                    current_streak = 0
            ghost_streaks[sender] = max_streak

    # Print the report section
    print("=" * 65)
    print("RESPONSE SPEED & SILENT STREAKS (GHOST DETECTOR)")
    print("=" * 65)
    print(f"{'Participant':<18} | {'Avg Response Time':<18} | {'Max Ghost Streak':<15}")
    print("-" * 65)

    participants = sorted(list(set(msg['sender'] for msg in chat_data)))
    for p in participants:
        avg_time = avg_response_times.get(p, 0.0)
        # Format time display
        if avg_time < 60:
            time_str = f"{avg_time:.1f} mins"
        else:
            time_str = f"{avg_time / 60:.1f} hours"

        ghost_days = ghost_streaks.get(p, 0)
        ghost_str = f"{ghost_days} days"

        print(f"{p:<18} | {time_str:<18} | {ghost_str:<15}")

    print("=" * 65)

# Run Feature 6
analyze_response_speeds_and_streaks(chat_data)

RESPONSE SPEED & SILENT STREAKS (GHOST DETECTOR)
Participant        | Avg Response Time  | Max Ghost Streak
-----------------------------------------------------------------
Aman               | 55.4 mins          | 0 days         
Karan              | 36.6 mins          | 0 days         
Neha               | 39.4 mins          | 0 days         
Priya              | 42.0 mins          | 0 days         
Rahul              | 34.9 mins          | 0 days         
Vikas              | 46.3 mins          | 11 days        


In [16]:
# ==========================================
# Feature 7: Personality Archetype Detection
# GroupDNA - Unlox Academy Minor Project
# ==========================================

def detect_personality_archetypes(chat_data):
    if not chat_data:
        print("⚠️ No chat data found. Please make sure Feature 1 ran successfully!")
        return

    # 1. Gather raw metrics per participant
    participant_stats = {}

    total_group_messages = len(chat_data)
    participants = list(set(msg['sender'] for msg in chat_data))

    for p in participants:
        participant_stats[p] = {
            'msg_count': 0,
            'late_night_count': 0,  # 00:00 to 04:59
            'morning_count': 0,     # 06:00 to 09:59
            'question_count': 0,    # Total '?'
            'exclamation_count': 0, # Total '!'
            'total_words': 0,       # For average message length
            'media_count': 0
        }

    for msg in chat_data:
        sender = msg['sender']
        text = msg['text']
        timestamp_str = msg['timestamp'] # 'DD/MM/YY, HH:MM'

        stats = participant_stats[sender]
        stats['msg_count'] += 1

        if text == '<Media omitted>':
            stats['media_count'] += 1

        # Track word count
        words = text.split()
        stats['total_words'] += len(words)

        # Track punctuation
        stats['question_count'] += text.count('?')
        stats['exclamation_count'] += text.count('!')

        # Extract hour
        try:
            time_part = timestamp_str.split(',')[1].strip()
            hour = int(time_part.split(':')[0])

            if 0 <= hour <= 4:
                stats['late_night_count'] += 1
            elif 6 <= hour <= 9:
                stats['morning_count'] += 1
        except Exception:
            pass

    # 2. Assign Archetypes based on behavioral scoring rules
    # We will ensure unique or logical distribution of archetypes across participants
    assigned_archetypes = {}
    available_archetypes = [
        ("The Spammer", "Sends the highest volume of messages and keeps the chat alive."),
        ("The Night Owl", "Most active during late-night hours (12 AM - 5 AM)."),
        ("The Storyteller", "Writes long, detailed paragraphs and essays."),
        ("The Question Master", "Asks the most questions and initiates plans."),
        ("The Hype Man / Energizer", "Uses lots of exclamation marks and high-energy chat cues."),
        ("The Group Mom / Caretaker", "Early riser who checks in during morning hours."),
        ("The Media Mogul", "Shares the highest number of photos, videos, and media stubs."),
        ("The Quiet Lurker", "Sends short, concise messages and speaks only when necessary.")
    ]

    # Track which archetypes have been claimed
    claimed_archetypes = set()

    # Helper to find best available match
    def assign_best(participant, default_title, default_desc):
        for title, desc in available_archetypes:
            if title not in claimed_archetypes:
                claimed_archetypes.add(title)
                return title, desc
        return default_title, default_desc

    # Sort participants by message count to prioritize top chatters
    sorted_by_activity = sorted(participants, key=lambda x: participant_stats[x]['msg_count'], reverse=True)

    for p in sorted_by_activity:
        stats = participant_stats[p]

        # Rule 1: Night Owl (high late-night ratio)
        if stats['late_night_count'] >= 15 and "The Night Owl" not in claimed_archetypes:
            title, desc = "The Night Owl", "Most active during late-night hours (12 AM - 5 AM)."
            claimed_archetypes.add(title)

        # Rule 2: Spammer (highest message count)
        elif p == sorted_by_activity[0] and "The Spammer" not in claimed_archetypes:
            title, desc = "The Spammer", "Sends the highest volume of messages and keeps the chat alive."
            claimed_archetypes.add(title)

        # Rule 3: Storyteller (highest average words per message)
        elif (stats['total_words'] / max(1, stats['msg_count'])) > 8 and "The Storyteller" not in claimed_archetypes:
            title, desc = "The Storyteller", "Writes long, detailed paragraphs and essays."
            claimed_archetypes.add(title)

        # Rule 4: Question Master (most questions asked)
        elif stats['question_count'] >= 5 and "The Question Master" not in claimed_archetypes:
            title, desc = "The Question Master", "Asks the most questions and initiates plans."
            claimed_archetypes.add(title)

        # Rule 5: Hype Man (most exclamation marks)
        elif stats['exclamation_count'] >= 8 and "The Hype Man / Energizer" not in claimed_archetypes:
            title, desc = "The Hype Man / Energizer", "Uses lots of exclamation marks and high-energy chat cues."
            claimed_archetypes.add(title)

        # Rule 6: Group Mom (active in mornings)
        elif stats['morning_count'] >= 5 and "The Group Mom / Caretaker" not in claimed_archetypes:
            title, desc = "The Group Mom / Caretaker", "Early riser who checks in during morning hours."
            claimed_archetypes.add(title)

        # Rule 7: Media Mogul (shares lots of media)
        elif stats['media_count'] >= 5 and "The Media Mogul" not in claimed_archetypes:
            title, desc = "The Media Mogul", "Shares the highest number of photos, videos, and media stubs."
            claimed_archetypes.add(title)

        # Fallback / Lurker
        else:
            title, desc = assign_best(p, "The Quiet Lurker", "Sends short, concise messages and speaks only when necessary.")

        assigned_archetypes[p] = (title, desc)

    # Print the report section
    print("=" * 75)
    print("PERSONALITY ARCHETYPE DETECTION")
    print("=" * 75)

    for p in participants:
        title, desc = assigned_archetypes[p]
        print(f"👤 {p}")
        print(f"   ➔ Archetype : {title}")
        print(f"   ➔ Profile   : {desc}\n")

    print("=" * 75)

# Run Feature 7
detect_personality_archetypes(chat_data)

PERSONALITY ARCHETYPE DETECTION
👤 Aman
   ➔ Archetype : The Spammer
   ➔ Profile   : Sends the highest volume of messages and keeps the chat alive.

👤 Karan
   ➔ Archetype : The Storyteller
   ➔ Profile   : Writes long, detailed paragraphs and essays.

👤 Priya
   ➔ Archetype : The Question Master
   ➔ Profile   : Asks the most questions and initiates plans.

👤 Neha
   ➔ Archetype : The Hype Man / Energizer
   ➔ Profile   : Uses lots of exclamation marks and high-energy chat cues.

👤 Rahul
   ➔ Archetype : The Night Owl
   ➔ Profile   : Most active during late-night hours (12 AM - 5 AM).

👤 Vikas
   ➔ Archetype : The Group Mom / Caretaker
   ➔ Profile   : Early riser who checks in during morning hours.



In [17]:
# ==========================================
# Feature 8: The Final Report Dashboard
# GroupDNA - Unlox Academy Minor Project
# ==========================================

from datetime import datetime, timedelta

def generate_final_groupdna_report(chat_data):
    if not chat_data:
        print("⚠️ No chat data found. Please make sure Feature 1 ran successfully!")
        return

    # --- 1. Core Calculations ---
    total_messages = len(chat_data)
    participants = sorted(list(set(msg['sender'] for msg in chat_data)))

    # Message counts per person
    message_counts = {}
    for msg in chat_data:
        sender = msg['sender']
        message_counts[sender] = message_counts.get(sender, 0) + 1
    sorted_participants = sorted(message_counts.items(), key=lambda x: x[1], reverse=True)
    top_chatter = sorted_participants[0][0]

    # Date range
    first_ts_str = chat_data[0]['timestamp'].split(',')[0].strip()
    last_ts_str = chat_data[-1]['timestamp'].split(',')[0].strip()
    date_format = '%d/%m/%y'
    first_date = datetime.strptime(first_ts_str, date_format)
    last_date = datetime.strptime(last_ts_str, date_format)
    total_days = (last_date - first_date).days + 1

    # Busiest Day & Hour
    daily_counts = {}
    hourly_counts = {}
    for msg in chat_data:
        d_part, t_part = msg['timestamp'].split(',')
        d_part = d_part.strip()
        h_part = t_part.strip().split(':')[0]
        daily_counts[d_part] = daily_counts.get(d_part, 0) + 1
        hourly_counts[h_part] = hourly_counts.get(h_part, 0) + 1

    busiest_date_str = max(daily_counts, key=daily_counts.get)
    busiest_date_formatted = datetime.strptime(busiest_date_str, '%d/%m/%y').strftime('%d %B %Y')

    peak_hour_str = max(hourly_counts, key=hourly_counts.get)
    h_int = int(peak_hour_str)
    peak_hour_formatted = f"{h_int:02d}:00 - {(h_int+1)%24:02d}:00"

    # Top Word (Quick calculation)
    stop_words = {'the', 'a', 'an', 'and', 'in', 'to', 'of', 'it', 'is', 'i', 'that', 'you', 'he', 'was', 'for', 'on', 'are', 'as', 'with', 'his', 'they', 'at', 'be', 'this', 'have', 'from', 'or', 'had', 'by', 'not', 'media', 'omitted', 'hai', 'ki', 'bhi', 'kya'}
    word_counts = {}
    punctuation = '''!()-[]{};:'"\\,<>./?@#$%^&*_~+`='''
    for msg in chat_data:
        text = msg['text']
        if text in ('<Media omitted>', 'This message was deleted'):
            continue
        cleaned = "".join([c if c not in punctuation else " " for c in text.lower()])
        for w in cleaned.split():
            if len(w) > 2 and w not in stop_words:
                word_counts[w] = word_counts.get(w, 0) + 1
    top_word = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[0][0] if word_counts else "N/A"

    # --- 2. Render Beautiful Terminal Dashboard ---
    box_width = 68

    print("╔" + "═" * (box_width - 2) + "╗")
    print(f"║{'📊 GROUPDNA: THE ULTIMATE WHATSAPP WRAPPED 📊':^{box_width-2}}║")
    print("╠" + "═" * (box_width - 2) + "╣")
    print(f"║  Chat Name     : Hostel Bois 4ever{'':<33}║")
    print(f"║  Date Range    : {first_date.strftime('%d %b %Y')} to {last_date.strftime('%d %b %Y')} ({total_days} days){'':<14}║")
    print(f"║  Total Msgs    : {total_messages:,}{'':<46}║")
    print(f"║  Participants  : {len(participants)}{'':<49}║")
    print("╠" + "═" * (box_width - 2) + "╣")
    print(f"║{' 🏆 HALL OF FAME & PEAK STATS ':^66}║")
    print("╠" + "═" * (box_width - 2) + "╣")
    print(f"║  👑 Top Chatter    : {top_chatter:<43}║")
    print(f"║  🔥 Busiest Day    : {busiest_date_formatted:<43}║")
    print(f"║  ⏰ Peak Hour      : {peak_hour_formatted:<43}║")
    print(f"║  💬 Top Word       : '{top_word}'{'':<41}║")
    print("╠" + "═" * (box_width - 2) + "╣")
    print(f"║{' 👥 PARTICIPANT BREAKDOWN ':^66}║")
    print("╠" + "═" * (box_width - 2) + "╣")

    for sender, count in sorted_participants:
        pct = (count / total_messages) * 100
        bar = '█' * int(pct / 3)
        line_content = f"  {sender:<12} : {count:4d} msgs ({pct:4.1f}%) {bar}"
        print(f"║{line_content:<66}║")

    print("╚" + "═" * (box_width - 2) + "╝")

# Run Feature 8
generate_final_groupdna_report(chat_data)

╔══════════════════════════════════════════════════════════════════╗
║           📊 GROUPDNA: THE ULTIMATE WHATSAPP WRAPPED 📊            ║
╠══════════════════════════════════════════════════════════════════╣
║  Chat Name     : Hostel Bois 4ever                                 ║
║  Date Range    : 01 Apr 2024 to 30 May 2024 (60 days)              ║
║  Total Msgs    : 3,174                                              ║
║  Participants  : 6                                                 ║
╠══════════════════════════════════════════════════════════════════╣
║                   🏆 HALL OF FAME & PEAK STATS                    ║
╠══════════════════════════════════════════════════════════════════╣
║  👑 Top Chatter    : Rahul                                      ║
║  🔥 Busiest Day    : 04 May 2024                                ║
║  ⏰ Peak Hour      : 18:00 - 19:00                              ║
║  💬 Top Word       : 'how'                                         ║
╠═════════════════════════════